# Enhanced Multimodal RAG — All Best Practices Integrated

## Changes Overview
```
✅ CLIP ViT-Large-Patch14    → Higher accuracy image embeddings
✅ Image Caption Fallback     → GPT-4o captions stored as text embeddings  
✅ Table Summarization        → LLM summaries embedded alongside raw tables
✅ Smart Chunking Strategy    → by_title (structured) vs basic (scanned)
✅ Cross-Encoder Reranking    → BGE reranker re-scores all retrieved docs
```

---

## Step 1: Install Additional Dependencies

```bash
pip install langchain langchain-openai langchain-community
pip install unstructured[pdf] pdfminer.six pymupdf
pip install chromadb openai pillow
pip install transformers torch torchvision sentence-transformers
pip install "unstructured[all-docs]" pytesseract   # OCR for scanned PDFs
```

---

## Step 2: Configuration & Setup

```python
import os
import fitz
import base64
import uuid
import torch
import numpy as np
from io import BytesIO
from PIL import Image
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain.schema.messages import HumanMessage
from langchain.schema.retriever import BaseRetriever
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import CrossEncoder
from unstructured.partition.pdf import partition_pdf
import chromadb

os.environ["OPENAI_API_KEY"] = "your-openai-key"

# ─────────────────────────────────────────────
# Global Config
# ─────────────────────────────────────────────
@dataclass
class RAGConfig:
    # CLIP — upgraded to large for higher accuracy
    clip_model_name: str = "openai/clip-vit-large-patch14"  # ✅ Best Practice #3

    # Reranker
    reranker_model: str = "BAAI/bge-reranker-large"         # ✅ Best Practice #5

    # Chunking
    chunking_strategy: str = "auto"   # "auto" | "by_title" | "basic"  ✅ #4

    # Retrieval
    k_text: int = 5
    k_images: int = 3
    k_reranked: int = 4               # Final top-K after reranking

    # Caption fallback threshold (cosine similarity)
    clip_confidence_threshold: float = 0.22  # Below this → caption fallback ✅ #1

    # Table summarization
    summarize_tables: bool = True      # ✅ Best Practice #2
    max_table_summary_tokens: int = 512

    # Paths
    image_output_dir: str = "./extracted_images/"
    chroma_persist_dir: str = "./chroma_db/"
```

---

## Step 3: Smart PDF Extraction with Auto Chunking Strategy

```python
def detect_pdf_type(pdf_path: str) -> str:
    """
    ✅ Best Practice #4 — Detect if PDF is text-based or scanned.
    Scanned PDFs have very little extractable text → use 'basic' (OCR-friendly).
    Structured PDFs with headings → use 'by_title'.
    """
    doc = fitz.open(pdf_path)
    total_text = ""
    for page in doc:
        total_text += page.get_text()
    doc.close()

    avg_chars_per_page = len(total_text) / max(len(doc), 1)
    
    if avg_chars_per_page < 100:
        print("📷 Detected scanned PDF → using 'basic' chunking strategy (OCR mode)")
        return "basic"
    else:
        print("📝 Detected structured PDF → using 'by_title' chunking strategy")
        return "by_title"


def extract_pdf_elements(pdf_path: str, config: RAGConfig) -> Tuple[List[str], List[str]]:
    """
    Extract text chunks and raw tables.
    ✅ Best Practice #4: Auto-select chunking strategy based on PDF type.
    """
    strategy = config.chunking_strategy
    if strategy == "auto":
        strategy = detect_pdf_type(pdf_path)

    # Shared kwargs
    base_kwargs = dict(
        filename=pdf_path,
        extract_images_in_pdf=True,
        image_output_dir_path=config.image_output_dir,
        infer_table_structure=True,
    )

    if strategy == "by_title":
        # Structured PDFs: chunk by title/section
        elements = partition_pdf(
            **base_kwargs,
            chunking_strategy="by_title",
            max_characters=4000,
            new_after_n_chars=3800,
            combine_text_under_n_chars=2000,
        )
    else:
        # Scanned PDFs: use basic OCR-friendly extraction
        elements = partition_pdf(
            **base_kwargs,
            strategy="ocr_only",       # force OCR
            chunking_strategy="basic",
            max_characters=3000,
        )

    texts, tables = [], []
    for el in elements:
        el_type = str(type(el))
        if "Table" in el_type:
            tables.append(str(el))
        elif "CompositeElement" in el_type or "Text" in el_type:
            text = str(el).strip()
            if text:
                texts.append(text)

    print(f"   → Extracted {len(texts)} text chunks, {len(tables)} tables")
    return texts, tables


def extract_images_from_pdf(pdf_path: str) -> List[Dict]:
    """Extract all images with page metadata using PyMuPDF."""
    doc = fitz.open(pdf_path)
    images = []

    for page_num, page in enumerate(doc):
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            pil_img = Image.open(BytesIO(base_image["image"])).convert("RGB")

            # Skip tiny images (icons, decorations)
            if pil_img.width < 100 or pil_img.height < 100:
                continue

            buffered = BytesIO()
            pil_img.save(buffered, format="PNG")
            b64 = base64.b64encode(buffered.getvalue()).decode()

            images.append({
                "image": pil_img,
                "b64": b64,
                "page": page_num + 1,
                "index": img_index,
            })

    print(f"   → Extracted {len(images)} images")
    return images
```

---

## Step 4: Table Summarization Before Embedding

```python
class TableSummarizer:
    """
    ✅ Best Practice #2 — Summarize tables with LLM before embedding.
    Stores BOTH raw table HTML and the LLM summary; embeds the summary
    for better semantic retrieval.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.llm = ChatOpenAI(model=model, max_tokens=512, temperature=0)

    def summarize(self, raw_table: str) -> str:
        prompt = f"""You are a data analyst. Summarize the following table concisely.
Describe: key columns, notable values, trends, and any important insights.
Keep the summary under 150 words. Do NOT reproduce the raw table.

TABLE:
{raw_table}

SUMMARY:"""
        response = self.llm.invoke([HumanMessage(content=prompt)])
        return response.content.strip()

    def summarize_batch(self, tables: List[str]) -> List[Dict]:
        """Returns list of dicts with raw + summary for each table."""
        results = []
        for i, table in enumerate(tables):
            print(f"   Summarizing table {i+1}/{len(tables)}...")
            try:
                summary = self.summarize(table)
            except Exception as e:
                print(f"   ⚠️ Summarization failed for table {i}: {e}")
                summary = table[:500]  # fallback to truncated raw

            results.append({
                "raw": table,
                "summary": summary,
                "table_id": i
            })
        return results
```

---

## Step 5: CLIP Embedder (Upgraded to Large Model)

```python
class CLIPEmbedder:
    """
    ✅ Best Practice #3 — Uses clip-vit-large-patch14 for higher accuracy.
    Handles both image and text embeddings in the shared CLIP space.
    """

    def __init__(self, model_name: str = "openai/clip-vit-large-patch14"):
        print(f"   Loading CLIP model: {model_name} ...")
        self.model = CLIPModel.from_pretrained(model_name)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.model.eval()
        print(f"   ✅ CLIP loaded on {self.device}")

    def embed_image(self, image: Image.Image) -> List[float]:
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            feats = self.model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()[0].tolist()

    def embed_text(self, text: str) -> List[float]:
        """Text embedding in CLIP space — used for image retrieval."""
        inputs = self.processor(
            text=[text], return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            feats = self.model.get_text_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()[0].tolist()

    def cosine_similarity(self, a: List[float], b: List[float]) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
```

---

## Step 6: Image Captioning Fallback

```python
class ImageCaptionGenerator:
    """
    ✅ Best Practice #1 — If CLIP similarity is low (image is ambiguous
    or CLIP embedding is weak), generate a rich caption using GPT-4o.
    The caption is then stored as a TEXT embedding alongside the image,
    enabling semantic text-based image retrieval as a fallback.
    """

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o", max_tokens=300, temperature=0)

    def generate_caption(self, b64_image: str) -> str:
        prompt = [HumanMessage(content=[
            {
                "type": "text",
                "text": (
                    "Describe this image in detail for a RAG retrieval system. "
                    "Include: what is shown, any text visible, charts/graphs data, "
                    "colors, layout, and the likely context or topic. "
                    "Be specific and concise (max 100 words)."
                )
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{b64_image}", "detail": "high"}
            }
        ])]
        response = self.llm.invoke(prompt)
        return response.content.strip()

    def generate_captions_batch(
        self,
        images: List[Dict],
        clip_embedder: CLIPEmbedder,
        threshold: float
    ) -> List[Dict]:
        """
        For each image, check CLIP confidence via a generic probe query.
        If below threshold → generate GPT-4o caption.
        """
        probe_queries = [
            "a chart or graph",
            "a diagram or figure",
            "a photograph or illustration",
            "text or table in an image"
        ]

        enriched = []
        for i, img_data in enumerate(images):
            img_embedding = clip_embedder.embed_image(img_data["image"])

            # Check max similarity against probe queries
            max_sim = max(
                clip_embedder.cosine_similarity(
                    img_embedding,
                    clip_embedder.embed_text(q)
                )
                for q in probe_queries
            )

            caption = None
            if max_sim < threshold:
                print(f"   🔄 Image {i+1}: CLIP confidence {max_sim:.3f} < {threshold} → generating GPT-4o caption")
                try:
                    caption = generate_caption_for_image(img_data["b64"])
                except Exception as e:
                    print(f"   ⚠️ Caption failed: {e}")
                    caption = "An image from the document."
            else:
                print(f"   ✅ Image {i+1}: CLIP confidence {max_sim:.3f} — no caption needed")

            enriched.append({
                **img_data,
                "clip_embedding": img_embedding,
                "caption": caption,
                "clip_confidence": max_sim,
                "use_caption_fallback": caption is not None
            })

        return enriched


# Standalone caption helper (used inside batch above)
_caption_llm = None

def generate_caption_for_image(b64_image: str) -> str:
    global _caption_llm
    if _caption_llm is None:
        _caption_llm = ChatOpenAI(model="gpt-4o", max_tokens=300, temperature=0)

    response = _caption_llm.invoke([HumanMessage(content=[
        {"type": "text", "text": (
            "Describe this image for a document retrieval system. "
            "Include visible text, charts, diagrams, key data, and context. "
            "Be specific (max 100 words)."
        )},
        {"type": "image_url", "image_url": {
            "url": f"data:image/png;base64,{b64_image}", "detail": "high"
        }}
    ])])
    return response.content.strip()
```

---

## Step 7: Build Unified Vector Stores

```python
def build_text_table_vectorstore(
    texts: List[str],
    table_data: List[Dict],   # output from TableSummarizer
    persist_dir: str
) -> Chroma:
    """
    ✅ Best Practice #2 integrated: Embeds TABLE SUMMARIES (not raw tables).
    Raw table is kept in metadata for final context injection.
    """
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
    docs = []

    # Text chunks
    for i, text in enumerate(texts):
        docs.append(Document(
            page_content=text,
            metadata={"type": "text", "chunk_id": i}
        ))

    # Tables — embed summary, store raw in metadata
    for td in table_data:
        docs.append(Document(
            page_content=td["summary"],          # ✅ embed summary
            metadata={
                "type": "table",
                "table_id": td["table_id"],
                "raw_table": td["raw"][:2000],   # store raw (truncated for metadata limit)
                "summary": td["summary"]
            }
        ))

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        collection_name="text_table_store",
        persist_directory=persist_dir
    )
    print(f"   ✅ Text+Table vectorstore: {len(docs)} documents indexed")
    return vectorstore


def build_image_vectorstore(
    enriched_images: List[Dict],
    clip_embedder: CLIPEmbedder,
    text_embedding_model: OpenAIEmbeddings,
    persist_dir: str
) -> Tuple[chromadb.Collection, Chroma]:
    """
    ✅ Best Practice #1 integrated:
    - Primary CLIP collection for image↔text matching
    - Fallback caption collection (OpenAI text embeddings) for weak CLIP images
    """
    # --- Primary CLIP image collection ---
    chroma_client = chromadb.PersistentClient(path=persist_dir)
    
    # Remove if exists (for re-indexing)
    try:
        chroma_client.delete_collection("clip_image_store")
    except:
        pass

    clip_collection = chroma_client.create_collection(
        name="clip_image_store",
        metadata={"hnsw:space": "cosine"}
    )

    ids, embeddings, metadatas = [], [], []
    caption_docs = []

    for img_data in enriched_images:
        img_id = str(uuid.uuid4())
        ids.append(img_id)
        embeddings.append(img_data["clip_embedding"])
        metadatas.append({
            "page": img_data["page"],
            "b64_image": img_data["b64"],
            "type": "image",
            "has_caption": img_data["use_caption_fallback"],
            "clip_confidence": round(img_data["clip_confidence"], 4),
            "caption": img_data.get("caption") or "",
            "image_id": img_id
        })

        # ✅ Build caption fallback docs for text-embedding-based search
        if img_data["use_caption_fallback"] and img_data.get("caption"):
            caption_docs.append(Document(
                page_content=img_data["caption"],
                metadata={
                    "type": "image_caption",
                    "image_id": img_id,
                    "page": img_data["page"],
                    "b64_image": img_data["b64"],
                    "caption": img_data["caption"]
                }
            ))

    if ids:
        clip_collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=["[IMAGE]"] * len(ids))
    print(f"   ✅ CLIP image collection: {len(ids)} images indexed")

    # Caption fallback vectorstore (OpenAI embeddings)
    caption_vectorstore = None
    if caption_docs:
        caption_vectorstore = Chroma.from_documents(
            documents=caption_docs,
            embedding=text_embedding_model,
            collection_name="caption_fallback_store",
            persist_directory=persist_dir
        )
        print(f"   ✅ Caption fallback store: {len(caption_docs)} captions indexed")
    else:
        print("   ℹ️ No caption fallback needed (all images had good CLIP confidence)")

    return clip_collection, caption_vectorstore
```

---

## Step 8: Cross-Encoder Reranker

```python
class CrossEncoderReranker:
    """
    ✅ Best Practice #5 — Reranks all retrieved docs (text + tables + images)
    using a cross-encoder for better final context quality.
    Images use their caption or a generic description for scoring.
    """

    def __init__(self, model_name: str = "BAAI/bge-reranker-large"):
        print(f"   Loading reranker: {model_name} ...")
        self.reranker = CrossEncoder(model_name, max_length=512)
        print("   ✅ Reranker loaded")

    def rerank(self, query: str, docs: List[Document], top_k: int) -> List[Document]:
        if not docs:
            return []

        # Build (query, passage) pairs
        pairs = []
        for doc in docs:
            if doc.metadata.get("type") == "image":
                # Use caption if available, else generic description
                passage = doc.metadata.get("caption") or \
                          f"Image from page {doc.metadata.get('page', '?')} of the document."
            elif doc.metadata.get("type") == "table":
                # Use summary for reranking
                passage = doc.metadata.get("summary") or doc.page_content
            else:
                passage = doc.page_content
            pairs.append((query, passage[:512]))

        scores = self.reranker.predict(pairs)

        # Attach scores and sort
        scored_docs = sorted(
            zip(scores, docs),
            key=lambda x: x[0],
            reverse=True
        )

        top_docs = [doc for _, doc in scored_docs[:top_k]]
        print(f"   🔁 Reranked {len(docs)} → top {len(top_docs)} docs")
        for score, doc in scored_docs[:top_k]:
            dtype = doc.metadata.get("type", "text")
            print(f"      [{dtype}] score={score:.4f} | {doc.page_content[:60]}...")

        return top_docs
```

---

## Step 9: Unified Multimodal Retriever (All Best Practices)

```python
class MultimodalRetriever(BaseRetriever):
    """
    Unified retriever combining:
    ✅ #1 Caption fallback for weak CLIP images
    ✅ #3 CLIP-large for primary image retrieval
    ✅ #5 Cross-encoder reranking on final result set
    """

    text_vectorstore: object
    clip_collection: object
    caption_vectorstore: Optional[object]
    clip_embedder: object
    reranker: object
    config: RAGConfig

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:

        docs = []

        # ── 1. Text + Table retrieval (OpenAI embeddings) ──────────────
        text_docs = self.text_vectorstore.similarity_search(
            query, k=self.config.k_text
        )
        docs.extend(text_docs)
        print(f"   📄 Text/table retrieved: {len(text_docs)} docs")

        # ── 2. Primary CLIP image retrieval ────────────────────────────
        query_clip_embedding = self.clip_embedder.embed_text(query)
        image_results = self.clip_collection.query(
            query_embeddings=[query_clip_embedding],
            n_results=min(self.config.k_images, self.clip_collection.count() or 1),
            include=["metadatas", "distances"]
        )

        clip_image_ids = set()
        if image_results and image_results["metadatas"]:
            for meta in image_results["metadatas"][0]:
                clip_image_ids.add(meta.get("image_id"))
                docs.append(Document(
                    page_content=meta.get("caption") or "[IMAGE]",
                    metadata={**meta, "type": "image"}
                ))
            print(f"   🖼️  CLIP images retrieved: {len(image_results['metadatas'][0])}")

        # ── 3. Caption fallback retrieval ✅ Best Practice #1 ──────────
        if self.caption_vectorstore:
            caption_docs = self.caption_vectorstore.similarity_search(
                query, k=self.config.k_images
            )
            # Only add if not already retrieved by CLIP
            added = 0
            for cd in caption_docs:
                if cd.metadata.get("image_id") not in clip_image_ids:
                    docs.append(cd)
                    added += 1
            if added:
                print(f"   📝 Caption fallback retrieved: {added} additional images")

        # ── 4. Cross-encoder reranking ✅ Best Practice #5 ─────────────
        reranked = self.reranker.rerank(query, docs, top_k=self.config.k_reranked)

        return reranked
```

---

## Step 10: GPT-4o Multimodal Generation Chain

```python
def build_multimodal_prompt(query: str, docs: List[Document]) -> List[HumanMessage]:
    """Build a rich multimodal prompt with text, table summaries, and images."""
    
    content = []
    text_context = ""
    table_context = ""
    images_b64 = []

    for doc in docs:
        dtype = doc.metadata.get("type", "text")

        if dtype == "text":
            text_context += f"\n[TEXT]\n{doc.page_content}\n"

        elif dtype == "table":
            # Show summary + raw table for accuracy
            summary = doc.metadata.get("summary", doc.page_content)
            raw = doc.metadata.get("raw_table", "")
            table_context += f"\n[TABLE SUMMARY]\n{summary}\n"
            if raw:
                table_context += f"[RAW TABLE DATA]\n{raw[:1000]}\n"

        elif dtype in ("image", "image_caption"):
            b64 = doc.metadata.get("b64_image")
            if b64 and b64 not in images_b64:
                images_b64.append(b64)

    # Build system context block
    full_context = ""
    if text_context:
        full_context += f"=== TEXT CONTEXT ===\n{text_context}\n"
    if table_context:
        full_context += f"=== TABLE CONTEXT ===\n{table_context}\n"

    content.append({
        "type": "text",
        "text": f"""You are an expert assistant analyzing a document.
Answer the question using ALL provided context: text, tables, and images.
Be accurate, cite which source supports your answer (text/table/image).

{full_context}

Question: {query}

Answer:"""
    })

    # Attach images
    for b64 in images_b64[:4]:   # GPT-4o supports up to 4 images well
        content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{b64}",
                "detail": "high"
            }
        })

    return [HumanMessage(content=content)]


def create_rag_chain(retriever: MultimodalRetriever):
    llm = ChatOpenAI(model="gpt-4o", max_tokens=2000, temperature=0)

    def rag_chain(query: str) -> Dict:
        print(f"\n🔍 Query: {query}")
        print("─" * 60)

        retrieved = retriever.get_relevant_documents(query)
        messages = build_multimodal_prompt(query, retrieved)
        response = llm.invoke(messages)

        return {
            "query": query,
            "answer": response.content,
            "retrieved_docs": retrieved,
            "num_text": sum(1 for d in retrieved if d.metadata.get("type") == "text"),
            "num_tables": sum(1 for d in retrieved if d.metadata.get("type") == "table"),
            "num_images": sum(1 for d in retrieved if "image" in d.metadata.get("type", "")),
        }

    return rag_chain
```

---

## Step 11: Master Pipeline — Full Integration

```python
def build_multimodal_rag_pipeline(pdf_path: str, config: RAGConfig = None) -> callable:
    """
    Complete pipeline with all 5 best practices:
    #1 Caption fallback  #2 Table summarization  #3 CLIP-large
    #4 Smart chunking    #5 Cross-encoder reranking
    """
    config = config or RAGConfig()
    text_embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

    # ── Extract ────────────────────────────────────────────────────────
    print("\n📄 STEP 1: Extracting PDF elements...")
    texts, raw_tables = extract_pdf_elements(pdf_path, config)
    images = extract_images_from_pdf(pdf_path)

    # ── Table Summarization ✅ #2 ────────────────────────────────────
    print("\n📊 STEP 2: Summarizing tables...")
    if config.summarize_tables and raw_tables:
        summarizer = TableSummarizer()
        table_data = summarizer.summarize_batch(raw_tables)
    else:
        table_data = [
            {"raw": t, "summary": t[:500], "table_id": i}
            for i, t in enumerate(raw_tables)
        ]

    # ── CLIP Embedder ✅ #3 ─────────────────────────────────────────
    print(f"\n🔢 STEP 3: Loading CLIP ({config.clip_model_name})...")
    clip_embedder = CLIPEmbedder(model_name=config.clip_model_name)

    # ── Caption Fallback ✅ #1 ──────────────────────────────────────
    print("\n🖼️  STEP 4: Embedding images with CLIP + caption fallback...")
    caption_gen = ImageCaptionGenerator()
    enriched_images = caption_gen.generate_captions_batch(
        images, clip_embedder, threshold=config.clip_confidence_threshold
    )

    # ── Build Vector Stores ─────────────────────────────────────────
    print("\n🗄️  STEP 5: Building vector stores...")
    text_vs = build_text_table_vectorstore(texts, table_data, config.chroma_persist_dir)
    clip_collection, caption_vs = build_image_vectorstore(
        enriched_images, clip_embedder, text_embedding_model, config.chroma_persist_dir
    )

    # ── Reranker ✅ #5 ──────────────────────────────────────────────
    print("\n⚖️  STEP 6: Loading cross-encoder reranker...")
    reranker = CrossEncoderReranker(model_name=config.reranker_model)

    # ── Assemble Retriever ──────────────────────────────────────────
    retriever = MultimodalRetriever(
        text_vectorstore=text_vs,
        clip_collection=clip_collection,
        caption_vectorstore=caption_vs,
        clip_embedder=clip_embedder,
        reranker=reranker,
        config=config
    )

    rag_chain = create_rag_chain(retriever)
    print("\n✅ Multimodal RAG pipeline ready!\n")
    return rag_chain


# ──────────────────────────────────────────
# Run It
# ──────────────────────────────────────────
if __name__ == "__main__":
    config = RAGConfig(
        clip_model_name="openai/clip-vit-large-patch14",   # #3
        reranker_model="BAAI/bge-reranker-large",          # #5
        chunking_strategy="auto",                          # #4
        summarize_tables=True,                             # #2
        clip_confidence_threshold=0.22,                    # #1
        k_text=5,
        k_images=3,
        k_reranked=4,
    )

    rag = build_multimodal_rag_pipeline("your_document.pdf", config)

    # Test queries
    queries = [
        "What does the revenue chart show for Q3?",
        "Summarize the key findings from the data tables",
        "What is shown in the architecture diagram?",
    ]

    for q in queries:
        result = rag(q)
        print(f"\n{'='*60}")
        print(f"Q: {result['query']}")
        print(f"A: {result['answer']}")
        print(f"Sources used → Text: {result['num_text']} | Tables: {result['num_tables']} | Images: {result['num_images']}")
```

---

## Complete Best Practices Summary

| # | Practice | Where Applied | Impact |
|---|---|---|---|
| **#1** | Caption fallback via GPT-4o | `ImageCaptionGenerator` + `caption_vectorstore` | Recovers retrieval for ambiguous/complex images |
| **#2** | LLM table summarization | `TableSummarizer` → embedded as `page_content` | Semantic search works on meaning, not raw HTML |
| **#3** | CLIP ViT-Large-Patch14 | `CLIPEmbedder(model_name=...)` | ~15% better image retrieval vs base model |
| **#4** | Auto chunking strategy | `detect_pdf_type()` → `by_title` or `basic` | Correct handling for both native and scanned PDFs |
| **#5** | Cross-encoder reranking | `CrossEncoderReranker` → final top-K | Removes irrelevant retrieved docs before LLM call |